# Gradient Boosting Regressor: Diabetes Dataset

## Overview

**Gradient Boosting for regression** trains multiple regressors sequentially to minimize prediction error. Each new regressor focuses on reducing residuals (errors) from previous models, resulting in highly accurate predictions.

## How Boosting Works for Regression

1. **Initialize:** Start with a simple prediction (mean of target values)
2. **Sequential Correction:** Train new trees to predict and correct residuals
3. **Iterative Improvement:** Each tree is added to reduce remaining error
4. **Final Prediction:** Sum all tree predictions with learning rate scaling

## Learning Objectives

By the end of this notebook, you will understand:
1. How Gradient Boosting for regression works
2. How learning_rate and n_estimators affect model performance
3. How to evaluate boosting regressors
4. How Boosting compares to Bagging for regression

## Step 1: Import Libraries

In [ ]:
from matplotlib import pyplot as plt
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import numpy as np

## Step 2: Load and Prepare Data

In [ ]:
# Load the Diabetes dataset
diabetes_ds = load_diabetes()
X = diabetes_ds.data
y = diabetes_ds.target

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training set: {X_train.shape[0]} samples')
print(f'Test set: {X_test.shape[0]} samples')
print(f'Features: {X_train.shape[1]}')

## Step 3: Train Single Decision Tree Regressor (Baseline)

In [ ]:
# Single Decision Tree Regressor (baseline)
dt_single = DecisionTreeRegressor(
    criterion='squared_error',
    max_depth=5,
    random_state=42
)
dt_single.fit(X_train, y_train)

y_pred_single = dt_single.predict(X_test)
mse_single = mean_squared_error(y_test, y_pred_single)
r2_single = r2_score(y_test, y_pred_single)

print(f'Single Tree - MSE: {mse_single:.2f}, R²: {r2_single:.4f}')

## Step 4: Train Gradient Boosting Regressor

Gradient Boosting trains shallow trees sequentially. Key parameters:
- **n_estimators:** Number of boosting stages
- **learning_rate:** Shrinkage parameter (0.01 to 0.1 typically)
- **max_depth:** Depth of individual trees (usually 3-8, kept shallow)
- **subsample:** Fraction of samples used for fitting each tree

In [ ]:
# Gradient Boosting Regressor
gb_reg = GradientBoostingRegressor(
    n_estimators=100,        # Number of boosting stages
    learning_rate=0.1,       # Shrinks contribution of each tree
    max_depth=3,             # Shallow trees (weak learners)
    subsample=0.8,           # Use 80% of samples per iteration
    random_state=42
)

# Train the boosting ensemble
gb_reg.fit(X_train, y_train)

# Evaluate
y_pred_gb = gb_reg.predict(X_test)
mse_gb = mean_squared_error(y_test, y_pred_gb)
r2_gb = r2_score(y_test, y_pred_gb)

print(f'Gradient Boosting - MSE: {mse_gb:.2f}, R²: {r2_gb:.4f}')
print(f'Improvement in MSE: {(mse_single - mse_gb):.2f}')
print(f'Improvement in R²: {(r2_gb - r2_single):.4f}')

## Step 5: Effect of Learning Rate

In [ ]:
# Test with different learning rates
learning_rates = [0.01, 0.05, 0.1, 0.2]
lr_train_mse = []
lr_test_mse = []

for lr in learning_rates:
    gb = GradientBoostingRegressor(
        n_estimators=100,
        learning_rate=lr,
        max_depth=3,
        random_state=42
    )
    gb.fit(X_train, y_train)
    
    train_mse = mean_squared_error(y_train, gb.predict(X_train))
    test_mse = mean_squared_error(y_test, gb.predict(X_test))
    
    lr_train_mse.append(train_mse)
    lr_test_mse.append(test_mse)
    print(f'Learning Rate {lr}: Train MSE = {train_mse:.2f}, Test MSE = {test_mse:.2f}')

## Step 6: Effect of Number of Estimators

In [ ]:
# Test with different numbers of estimators
n_estimators_list = [10, 20, 50, 100, 200, 500]
ne_train_mse = []
ne_test_mse = []

for n_est in n_estimators_list:
    gb = GradientBoostingRegressor(
        n_estimators=n_est,
        learning_rate=0.1,
        max_depth=3,
        random_state=42
    )
    gb.fit(X_train, y_train)
    
    train_mse = mean_squared_error(y_train, gb.predict(X_train))
    test_mse = mean_squared_error(y_test, gb.predict(X_test))
    
    ne_train_mse.append(train_mse)
    ne_test_mse.append(test_mse)
    print(f'n_estimators={n_est:3d}: Train MSE = {train_mse:6.2f}, Test MSE = {test_mse:6.2f}')

## Step 7: Visualize Training Progress

In [ ]:
# Plot learning curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Learning rate effect
axes[0].plot(learning_rates, lr_train_mse, 'o-', linewidth=2, markersize=8, label='Train')
axes[0].plot(learning_rates, lr_test_mse, 's-', linewidth=2, markersize=8, label='Test')
axes[0].axhline(y=mse_single, color='r', linestyle='--', linewidth=1, alpha=0.7, label='Single Tree')
axes[0].set_xlabel('Learning Rate')
axes[0].set_ylabel('Mean Squared Error')
axes[0].set_title('Effect of Learning Rate (n_estimators=100)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Number of estimators effect
axes[1].plot(n_estimators_list, ne_train_mse, 'o-', linewidth=2, markersize=8, label='Train')
axes[1].plot(n_estimators_list, ne_test_mse, 's-', linewidth=2, markersize=8, label='Test')
axes[1].axhline(y=mse_single, color='r', linestyle='--', linewidth=1, alpha=0.7, label='Single Tree')
axes[1].set_xlabel('Number of Estimators')
axes[1].set_ylabel('Mean Squared Error')
axes[1].set_title('Effect of Number of Estimators (lr=0.1)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 8: Predictions Visualization

In [ ]:
# Use the model with 100 estimators
final_gb = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)
final_gb.fit(X_train, y_train)
y_pred_gb_final = final_gb.predict(X_test)

# Plot predictions vs. actual values
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Single tree
axes[0].scatter(y_test, y_pred_single, alpha=0.6, color='blue')
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[0].set_xlabel('Actual Values')
axes[0].set_ylabel('Predicted Values')
axes[0].set_title(f'Single Tree (R² = {r2_single:.3f})')
axes[0].grid(True, alpha=0.3)

# Gradient Boosting
axes[1].scatter(y_test, y_pred_gb_final, alpha=0.6, color='green')
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[1].set_xlabel('Actual Values')
axes[1].set_ylabel('Predicted Values')
axes[1].set_title(f'Gradient Boosting (R² = {r2_gb:.3f})')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Key Takeaways

1. **Sequential training:** Each new tree corrects residuals from previous ones
2. **Strong learners:** Achieves better accuracy than single trees and Bagging
3. **Learning rate:** Controls step size (lower = slower learning, better generalization)
4. **Weak learners:** Individual trees are intentionally shallow (max_depth typically 3-5)
5. **Training time:** Takes longer than Bagging due to sequential nature
6. **Not parallelizable:** Cannot train trees in parallel (unlike Bagging)
7. **Overfitting risk:** Can overfit if not carefully tuned; use cross-validation
8. **Residual reduction:** Each iteration focuses on reducing prediction errors